# Automação de Sistemas e Processos com Python

## Contexto do Problema

Todo dia, o sistema interno da empresa atualiza automaticamente um arquivo Excel com as vendas do dia anterior e o disponibiliza em uma pasta no Google Drive.

Como analista, a sua tarefa diária é:
1. Baixar esse arquivo do Google Drive
2. Calcular os indicadores de vendas (faturamento e quantidade de produtos vendidos)
3. Enviar um e-mail para a diretoria com esses números

## Solução

Para automatizar esse processo repetitivo, usamos o **PyAutoGUI** — uma biblioteca Python que simula ações do mouse e do teclado, permitindo controlar o computador de forma programática, como se fosse um usuário humano navegando pela tela.

**Referências:**
- Documentação do PyAutoGUI: https://pyautogui.readthedocs.io/en/latest/quickstart.html
- Pasta de vendas no Google Drive: https://drive.google.com/drive/folders/149xknr9JvrlEnhNWO49zPcw0PW5icxga
- Destinatário do e-mail: seugmail+diretoria@gmail.com

> ⚠️ **Atenção:** As coordenadas de clique (`x`, `y`) usadas neste script são específicas para a resolução e layout da tela em que ele foi criado. Se você rodar em outro computador, precisará ajustá-las. Veja a última célula para saber como descobrir as coordenadas corretas na sua tela.

## Etapa 1 — Abrir o Google Drive e baixar o arquivo de vendas

O bloco abaixo:
1. Abre o navegador Chrome pelo menu Iniciar do Windows
2. Acessa a pasta do Google Drive onde o arquivo de vendas está disponível
3. Clica no arquivo Excel para selecioná-lo e em seguida aciona o download

O `pyautogui.PAUSE = 1` adiciona uma pausa de 1 segundo entre cada ação, evitando que os comandos sejam executados rápido demais antes da interface carregar.

In [7]:
import pyautogui
import time

pyautogui.PAUSE = 1

pyautogui.press("win")
pyautogui.write("chrome")
pyautogui.press("enter")

time.sleep(2)

site = "https://drive.google.com/drive/folders/14oLE59U1RqyRqlBbKpsyymW-mitvbtoh"

pyautogui.write(site)
pyautogui.press("enter")

time.sleep(2)

### Cliques para selecionar e baixar o arquivo

Após a pasta do Drive carregar, o script:
- Clica sobre o arquivo Excel de vendas para selecioná-lo
- Clica no botão de download para salvá-lo localmente

> 📌 As coordenadas abaixo (`x=817, y=510` e `x=1652, y=516`) correspondem às posições do arquivo e do botão de download **na tela original**. Ajuste conforme necessário.

In [8]:
pyautogui.click(x=817, y=510)
pyautogui.click(x=1652, y=516)

## Etapa 2 — Ler o arquivo baixado e calcular os indicadores de vendas

Com o arquivo salvo localmente, usamos o **pandas** para lê-lo e extrair dois indicadores principais:

| Indicador | Coluna no Excel | Operação |
|---|---|---|
| **Faturamento total** | `Valor Final` | Soma de todos os valores |
| **Quantidade de produtos vendidos** | `Quantidade` | Soma de todas as quantidades |

> 📁 O caminho do arquivo (`path`) está definido como o local padrão de download do Windows. Altere se necessário.

In [9]:
import pandas as pd

path = r"C:\Users\kcosta\Downloads\Vendas - Dez.xlsx"
table = pd.read_excel(path)
# display(table)

In [10]:
faturamento = table["Valor Final"].sum()
qtde_produtos = table["Quantidade"].sum()

print(faturamento)
print(qtde_produtos)

2917311
15227


## Etapa 3 — Redigir e enviar o e-mail pelo Gmail

Com os indicadores calculados, o script abre o Gmail no navegador e envia automaticamente um e-mail para a diretoria. O processo é:

1. Abre uma nova aba e acessa o Gmail
2. Clica em "Escrever" para abrir a janela de novo e-mail
3. Preenche o campo **Para** com o e-mail da diretoria
4. Preenche o **Assunto** com "Relatório de Vendas"
5. Cola o corpo do e-mail com os valores de faturamento e quantidade formatados
6. Clica em **Enviar**

> 💡 Usamos o **pyperclip** para copiar textos com acentos e caracteres especiais para a área de transferência e colá-los com `Ctrl+V`, evitando erros de digitação que o `pyautogui.write()` pode causar com esse tipo de caractere.

In [11]:
import pyperclip

pyautogui.hotkey("ctrl", "t")
pyautogui.write("https://mail.google.com/")
pyautogui.press("enter")

time.sleep(2)

pyautogui.click(x=154, y=221)

time.sleep(2)

pyautogui.write("k.accosta98@gmail.com")
pyautogui.press("tab")
pyautogui.press("tab")

pyperclip.copy("Relatório de Vendas")
pyautogui.hotkey("ctrl", "v")
pyautogui.press("tab")

texto = f"""
Prezados,

Segue o relatório de vendas de hoje.

Faturamento: R$ {faturamento:,.2f}
Quantidade de produtos vendidos: {qtde_produtos:,}

Qualquer dúvida estou à disposição.

Atenciosamente,
Kelvin Python
"""

pyperclip.copy(texto)
pyautogui.hotkey("ctrl", "v")

pyautogui.click(x=1202, y=990)

## Utilitário — Descobrir a posição do cursor na tela

Como as coordenadas de clique variam de acordo com a resolução e o layout de cada computador, use o código abaixo para identificar a posição correta de qualquer elemento na **sua** tela:

1. Execute a célula
2. Nos 3 segundos de espera, mova o mouse até o elemento desejado (ex: um botão, um arquivo)
3. O script imprime as coordenadas `(x, y)` exatas do cursor naquele momento

Substitua os valores de `x` e `y` nos `pyautogui.click()` pelas coordenadas obtidas aqui.

In [12]:
time.sleep(3)
print(pyautogui.position())

Point(x=730, y=358)
